### Resampling


In [1]:
import pandas as pd
import numpy as np

# what counts as good coverage
EXPECTED_PER_HOUR = 30      # 60 min / 2-min cadence
HOURLY_MIN_FRAC   = 0.5     # an hour is "covered" if it has >=50% of expected readings

rows = []
for path in sorted(CLEANED.glob("*.parquet")):
    name  = path.stem
    df    = pd.read_parquet(path)
    clean = df[df["is_clean"]].copy()
    if clean.empty:
        continue
    clean[DATE_COL] = pd.to_datetime(clean[DATE_COL])

    ser = clean.set_index(DATE_COL)
    per_hour = ser[PM_A].resample("h").count()      # clean readings per hour

    rows.append({
        "sensor": name,
        "clean_rows": len(clean),
        "span_days": (clean[DATE_COL].max() - clean[DATE_COL].min()).days,
        "median_readings_per_hour": int(per_hour[per_hour > 0].median()),
        "pct_hours_well_covered": round(
            (per_hour >= EXPECTED_PER_HOUR * HOURLY_MIN_FRAC).mean() * 100, 1),
        "pct_rows_with_RH": round(clean[RH_COL].notna().mean() * 100, 1),
    })

cov = pd.DataFrame(rows).sort_values("pct_hours_well_covered", ascending=False)
cov.to_csv(OUT_TABLES / "coverage_check.csv", index=False)

print(cov.to_string(index=False))
print("\n--- network summary ---")
print(f"median % hours well-covered: {cov['pct_hours_well_covered'].median():.1f}%")
print(f"median % rows with RH:       {cov['pct_rows_with_RH'].median():.1f}%")
print(f"sensors with usable RH (>50%): {(cov['pct_rows_with_RH'] > 50).sum()} of {len(cov)}")

NameError: name 'CLEANED' is not defined